LOAD THE DATASET

In [1]:
import json
with open("text_segmentation_dataset.json","r",encoding="utf-8") as f:
    data=json.load(f)

word_counts = data["word_counts"]
test_cases = data["test_cases"]

print(type(data))
print(len(data))
print(data.keys())


<class 'dict'>
3
dict_keys(['metadata', 'word_counts', 'test_cases'])


In [3]:
for key,value in data.items():
    print("key:",key)
    print("type:",type(value))

    if isinstance(value,dict):
        print("Keys:",list(value.keys())[:10])
    elif isinstance(value,list):
        print("Length:",len(value))
        print("First item:",value[0])

key: metadata
type: <class 'dict'>
Keys: ['vocabulary_size', 'test_case_count', 'total_corpus_words']
key: word_counts
type: <class 'dict'>
Keys: ['the', 'of', 'and', 'to', 'a', 'in', 'that', 'is', 'was', 'he']
key: test_cases
type: <class 'list'>
Length: 1000
First item: {'input': 'itthatthecitytakestepstothisproblem', 'ground_truth': 'it that the city take steps to this problem', 'word_count': 9}


GREEDY APPROACH

In [4]:
def greedy_segment(text,word_counts):
    words=[]
    i=0

    while i<len(text):
        best_word=None

        #try every possible word starting at position i
        for j in range(i+1,len(text)+1):
            word=text[i:j]

            if word in word_counts:
                best_word=word

        #if no word was found,then we take the next char as a word
        if best_word is None:
            best_word=text[i:i+1]
            i+=1
            words.append(best_word)
            #return None
        else:
            words.append(best_word)
            i+=len(best_word)
    return words

In [5]:
test_case=data["test_cases"][0]
text=test_case["input"]
prediction=greedy_segment(text,data["word_counts"])
print("Input:", text)
print("\nPrediction:", prediction)
print("\nGround Truth:", test_case["ground_truth"])


Input: itthatthecitytakestepstothisproblem

Prediction: ['it', 'that', 'the', 'city', 'takes', 't', 'e', 'p', 's', 'to', 'this', 'problem']

Ground Truth: it that the city take steps to this problem


DP APPROACH

It needs word probabilities
P(word)=count(word)​/total word count

In [6]:
import math
def word_log_probability(word,word_counts):
    total=sum(word_counts.values())
    return math.log(word_counts[word]/total)

In [7]:
def dp_segment(text,word_counts):
    n=len(text)
    #dp[i] will store the best log probability for the first i characters
    dp=[float("-inf")]*(n+1)
    dp[0]=0

    #backtrack[i] will store the index of the last word in the best segmentation for the first i characters
    backtrack=[-1]*(n+1)

    total=sum(word_counts.values())
    max_word_len=max((len(word) for word in word_counts), default=1)
    min_log_prob=min((math.log(count/total) for count in word_counts.values()), default=-20)
    unknown_log_prob=min_log_prob-5

    for i in range(1,n+1):
        # Fallback to a single character so reconstruction always terminates.
        dp[i]=dp[i-1]+unknown_log_prob
        backtrack[i]=i-1

        for j in range(max(0,i-max_word_len),i):
            word=text[j:i]
            if word in word_counts:
                prob=math.log(word_counts[word]/total)
                if dp[j]+prob>dp[i]:
                    dp[i]=dp[j]+prob
                    backtrack[i]=j

    #reconstruct the best segmentation
    words=[]
    i=n
    while i>0:
        j=backtrack[i]
        if j<0:
            j=i-1
        words.append(text[j:i])
        i=j

    words.reverse()
    return words

In [8]:
test_case = data["test_cases"][0]
text = test_case["input"]
ground_truth = test_case["ground_truth"].split()

dp_prediction = dp_segment(text, word_counts)

print("Input:")
print(text)

print("\nDP Prediction:")
print(dp_prediction)

print("\nGround Truth:")
print(ground_truth)

Input:
itthatthecitytakestepstothisproblem

DP Prediction:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']

Ground Truth:
['it', 'that', 'the', 'city', 'take', 'steps', 'to', 'this', 'problem']


EDIT DISTANCE

In [9]:
def edit_distance(predicted,ground_truth):
    if predicted is None:
        return len(ground_truth)

    m=len(predicted)
    n=len(ground_truth)

    dp=[[0]*(n+1) for _ in range(m+1)]

    #convert predicted[0:i] -> empty
    for i in range(m+1):
        dp[i][0]=i

    #convert empty -> ground_truth[0:j]
    for j in range(n+1):
        dp[0][j]=j

    for i in range(1,m+1):
        for j in range(1,n+1):
            if predicted[i-1]==ground_truth[j-1]:
                cost=0
            else:
                cost=1

            dp[i][j]=min( dp[i-1][j]+1, #deletion
                         dp[i][j-1]+1, #insertion
                         dp[i-1][j-1]+cost #substitution
            )

    return dp[m][n]

In [9]:
distance=edit_distance(dp_prediction,ground_truth)
print("Edit Distance:",distance)

Edit Distance: 0


ACCURACY
accuracy=no. of correctly matched words/no. of ground_truth words

In [10]:
def calculate_accuracy(predicted,ground_truth):
    if predicted is None:
        return 0.0

    m=len(predicted)
    n=len(ground_truth)

    #dp[i][j] stores minimum edit distance between first i predicted words and first j ground truth words
    dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1):
        dp[i][0]=i
    for j in range(n+1):
        dp[0][j]=j

    for i in range(1,m+1):
        for j in range(1,n+1):
            if predicted[i-1]==ground_truth[j-1]:
                cost=0
            else:
                cost=1

            dp[i][j]=min( dp[i-1][j]+1,dp[i][j-1]+1,dp[i-1][j-1]+cost)

    #reconstruct the alignment to count correct words
    i=m
    j=n
    correct=0
    while i>0 and j>0:
        if predicted[i-1]==ground_truth[j-1]:
            correct+=1
            i-=1
            j-=1
        elif dp[i][j]==dp[i-1][j]+1:
            i-=1
        elif dp[i][j]==dp[i][j-1]+1:
            j-=1
        else:
            i-=1
            j-=1 

    return correct / n if n > 0 else 0.0


In [11]:
accuracy = calculate_accuracy(
    dp_prediction,
    ground_truth
)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Accuracy: 1.0
Accuracy (%): 100.0


Evaluate all testcases

In [12]:
greedy_accuracies = []
dp_accuracies = []

greedy_edit_distances = []
dp_edit_distances = []

greedy_failures = 0
dp_failures = 0

for case in test_cases:

    text = case["input"]
    ground_truth = case["ground_truth"].split()

    # Greedy
    greedy_prediction = greedy_segment(
        text,
        word_counts
    )

    if greedy_prediction is None:
        greedy_failures += 1

    greedy_acc = calculate_accuracy(greedy_prediction,ground_truth)

    greedy_dist = edit_distance(
        greedy_prediction,
        ground_truth
    )

    greedy_accuracies.append(greedy_acc)
    greedy_edit_distances.append(greedy_dist)

    # DP
    dp_prediction = dp_segment(
        text,
        word_counts
    )

    if dp_prediction is None:
        dp_failures += 1

    dp_acc = calculate_accuracy(
        dp_prediction,
        ground_truth
    )

    dp_dist = edit_distance(
        dp_prediction,
        ground_truth
    )

    dp_accuracies.append(dp_acc)
    dp_edit_distances.append(dp_dist)

FINAL RESULTS

In [13]:
greedy_avg_accuracy = sum(greedy_accuracies) / len(greedy_accuracies)
dp_avg_accuracy = sum(dp_accuracies) / len(dp_accuracies)

greedy_avg_edit_distance = (
    sum(greedy_edit_distances)
    / len(greedy_edit_distances)
)

dp_avg_edit_distance = (
    sum(dp_edit_distances)
    / len(dp_edit_distances)
)


print("FINAL RESULTS")

print("\nGREEDY LONGEST-MATCH")
print("Average Accuracy:",
      greedy_avg_accuracy * 100, "%")
print("Average Edit Distance:",
      greedy_avg_edit_distance)
print("Failed segmentations:",
      greedy_failures)

print("\nDYNAMIC PROGRAMMING")

print("Average Accuracy:",
      dp_avg_accuracy * 100, "%")
print("Average Edit Distance:",
      dp_avg_edit_distance)
print("Failed segmentations:",
      dp_failures)

FINAL RESULTS

GREEDY LONGEST-MATCH
Average Accuracy: 91.67880591630596 %
Average Edit Distance: 1.26
Failed segmentations: 0

DYNAMIC PROGRAMMING
Average Accuracy: 99.47470418470418 %
Average Edit Distance: 0.037
Failed segmentations: 0


In [14]:

print(f"{'Method':<25} {'Accuracy':<15} {'Edit Distance':<15}")


print(
    f"{'Greedy Longest Match':<25} "
    f"{greedy_avg_accuracy * 100:.2f}%"
    f"{'':<8}"
    f"{greedy_avg_edit_distance:.2f}"
)

print(
    f"{'Dynamic Programming':<25} "
    f"{dp_avg_accuracy * 100:.2f}%"
    f"{'':<8}"
    f"{dp_avg_edit_distance:.2f}"
)


Method                    Accuracy        Edit Distance  
Greedy Longest Match      91.68%        1.26
Dynamic Programming       99.47%        0.04
